In [ ]:
import numpy as np


class Ackley:
    def __init__(self):
        self.a = 20.0
        self.b = 0.2
        self.c = 2 * np.pi

    def __call__(self, x):
        x = 32.768 * x
        y = (
            -self.a * np.exp(-self.b * np.sqrt((x**2).mean(axis=1)))
            - np.exp(np.cos(self.c * x).mean(axis=1))
            + self.a
            + np.e
        )
        return -y

# TuRBO-ENN

**TURBO_ZERO**  
Final best value: -6.424210
Mean proposal time: 0.03 seconds

**TURBO_ENN**  
Final best value: -3.777176
Mean proposal time: 0.08 seconds

**TURBO_ONE**
[Email david.sweet@yu.edu with your results.]

In [ ]:
import time

from enn import TurboMode, TurboOptimizer

num_dim = 100
num_iterations = 300
num_arms = 100
bounds = np.array([[-1, 1]] * num_dim, dtype=float)

ackley = Ackley()

rng = np.random.default_rng(42)
optimizer = TurboOptimizer(
    bounds=bounds,
    # Regular TuRBO
    # mode=TurboMode.TURBO_ONE,
    # TuRBO w/no surrogate
    # mode=TurboMode.TURBO_ZERO,
    # TuRBO w/ENN surrogate
    mode=TurboMode.TURBO_ENN,
    num_arms=num_arms,
    rng=rng,
    hnsw_threshold=1000,
)

best_values = []
proposal_times = []
best_y = -np.inf

for iteration in range(num_iterations):
    t_0 = time.time()
    x_candidates = optimizer.ask(num_arms=num_arms)
    t_1 = time.time()
    proposal_times.append(t_1 - t_0)

    y_values = ackley(x_candidates)

    optimizer.tell(x_candidates, y_values)

    current_best = float(np.max(y_values))
    if current_best > best_y:
        best_y = current_best
    best_values.append(best_y)

    if (iteration + 1) % 10 == 0:
        print(f"Iteration {iteration + 1}/{num_iterations}: Best value = {best_y:.6f}")

print(f"\nFinal best value: {best_y:.6f}")
print(f"Mean proposal time: {np.mean(proposal_times):.2f} seconds")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(best_values, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Best Function Value")
plt.title("TurboOptimizer Convergence on 200D Ackley Function")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(proposal_times, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Proposal Time (seconds)")
plt.title("TurboOptimizer Proposal Times vs. Iteration")
plt.grid(True, alpha=0.3)
c = plt.axis()
plt.axis([0, num_iterations, 0, max(c[3], 1)])
plt.tight_layout()
plt.show()